# CrisisQ — Controlled runtime matrix (v1.1 — Kaggle environment fix)

**Frozen models, no retraining, no clinicians.** This is a planned numerical follow-up informed by the observed QUInt8 disagreement. It is not a new independent dataset or a claim of a new quantization algorithm.

**Run in Kaggle:** Internet ON; GPU not required; attach your latest **whole model-containing notebook output** under `/kaggle/input`, then Run All. No input ZIP is required. No model download or threshold fitting is performed.

The fixed suite compares ONNX Runtime **1.28.0 and 1.31.1**, four graph-optimization levels and one/four intra-op threads on identical graphs and inputs. The primary pair uses all 1,700 frozen messages. Additional complete jointly accepted CE/SAQS pairs, if actually present, use the same deterministic 256-message diagnostic panel. Missing seeds are reported rather than invented.

The primary pair gives **32 jobs**. With all seven complete seed pairs the suite has **448 jobs**, so allow a substantial CPU session. Jobs checkpoint every 64 cases. The primary jobs run first; numerical results are reported without selecting favorable settings.


## 1. Configuration — set before the first run

Keep the defaults for the first study. After inference starts, changing configuration or mounted model coverage requires **new output and temporary folders**. Do not combine results from different Kaggle hosts as one controlled suite.

`all_available_joint_pairs` uses every complete jointly accepted pair it finds. Choose `primary_only` before the first run only if you intentionally want the smaller 32-job study; its report cannot support multiseed claims.


In [ ]:
from pathlib import Path
import json,sys,os
assert sys.version_info >= (3,11), 'Use Python 3.11 or later with matching binary wheels.'
assert Path('/kaggle/input').exists(), 'Import this notebook in Kaggle and attach the model-containing output.'
CONFIG = {
    'input_root': '/kaggle/input',
    'output_root': '/kaggle/working/crisisq_runtime_matrix_v1_1',
    'temporary_root': '/kaggle/temp/crisisq_runtime_matrix_v1_1',
    'model_scope': 'all_available_joint_pairs',
    'runtime_versions': ['1.28.0', '1.31.1'],
    'optimization_levels': ['ORT_DISABLE_ALL', 'ORT_ENABLE_BASIC', 'ORT_ENABLE_EXTENDED', 'ORT_ENABLE_ALL'],
    'threads': [1, 4],
    'job_timeout_seconds': 5400,
}
OUTPUT = Path(CONFIG['output_root']); OUTPUT.mkdir(parents=True,exist_ok=True)
TEMPORARY = Path(CONFIG['temporary_root']); TEMPORARY.mkdir(parents=True,exist_ok=True)
BUNDLE = Path('/kaggle/working/crisisq_runtime_matrix_code_v1_1'); BUNDLE.mkdir(parents=True,exist_ok=True)
print('CPU-only inference. Python:',sys.version.split()[0], 'logical CPUs:',os.cpu_count())
print('Output:',OUTPUT)


## 2. Unpack the embedded, hash-checked code and metadata

This cell contains helper source, the frozen selection/membership recipe and text-free historical device endpoints. It contains **no model weights or raw source posts**. Helpers are written to the code folder for inspection. The pinned public source is reconstructed only if the exact full-case packet is missing.


In [ ]:
import base64,hashlib,io,zipfile
PAYLOAD_SHA256 = 'a50a88cb3e3febf28c511621254a44af0792240050cec9618bc0236ab53c9917'
PAYLOAD_BASE64 = '...'
raw = base64.b64decode(PAYLOAD_BASE64)
assert hashlib.sha256(raw).hexdigest()==PAYLOAD_SHA256
allowed = ['environment.py', 'study.py', 'worker.py', 'analyze.py', 'build_full_device_packet.py', 'full_evaluation_seed.json.gz', 'external_frozen.json', 'selection_frozen.json', 'historical_android_primary.json', 'HOW_TO_RUN.md']
with zipfile.ZipFile(io.BytesIO(raw)) as archive:
    assert set(archive.namelist())==set(allowed)
    for name in allowed:
        assert Path(name).name==name
        (BUNDLE/name).write_bytes(archive.read(name))
del raw, PAYLOAD_BASE64
print('Verified and unpacked',len(allowed),'helper/data files into',BUNDLE)


## 3. Prepare the exact cases and freeze the protocol before inference

Environments are created without `ensurepip`, and the host pip manages only their isolated interpreters through `--python`. Preparation runs in a separate environment using NumPy 2.1.3 and tokenizers 0.23.3. The notebook kernel's packages are not downgraded. This step verifies graph hashes, source membership and token bytes, then saves the immutable protocol, model coverage and job order. It does not run a classifier.


In [ ]:
import subprocess,importlib.util,traceback
env_spec=importlib.util.spec_from_file_location('crisisq_environments',BUNDLE/'environment.py')
envtools=importlib.util.module_from_spec(env_spec);env_spec.loader.exec_module(envtools)
PREP_ENV = TEMPORARY/'preparation_environment'
PREP_PYTHON = envtools.ensure_environment(PREP_ENV)
probe=subprocess.run([str(PREP_PYTHON),'-I','-c','import numpy,tokenizers;print(numpy.__version__+"|"+tokenizers.__version__)'],capture_output=True,text=True)
if probe.returncode or probe.stdout.strip()!='2.1.3|0.23.3':
    log=OUTPUT/'preparation_install.log'
    print('Installing isolated preparation dependencies; Internet must be ON.',flush=True)
    with log.open('w') as handle:
        status=subprocess.run(envtools.pip_command(PREP_PYTHON,'install','--only-binary=:all:','numpy==2.1.3','tokenizers==0.23.3'),stdout=handle,stderr=subprocess.STDOUT)
    if status.returncode:
        print(log.read_text()[-2500:])
        raise RuntimeError('Preparation wheels unavailable for this Python image. No version substitution is allowed.')
CONFIG_PATH=TEMPORARY/'configuration.json';CONFIG_PATH.write_text(json.dumps(CONFIG,indent=2))
subprocess.run([str(PREP_PYTHON),'-I',str(BUNDLE/'study.py'),'--prepare',str(CONFIG_PATH)],check=True)
def load_helper(name):
    spec=importlib.util.spec_from_file_location('crisisq_matrix_'+name,BUNDLE/(name+'.py'))
    module=importlib.util.module_from_spec(spec);spec.loader.exec_module(module);return module
study=load_helper('study')
frozen=json.loads((OUTPUT/'protocol_frozen.json').read_text())
PROTOCOL=frozen['protocol'];JOBS=json.loads((OUTPUT/'jobs_frozen.json').read_text())
print('Frozen plan:',frozen['plan_hash'])
print('Artifacts:',len(PROTOCOL['models']),'settings:',len(PROTOCOL['settings']),'jobs:',len(JOBS))
print('Jointly accepted AND available CE/SAQS pairs:',sum(r['included'] for r in PROTOCOL['coverage']))
for item in PROTOCOL['coverage']:print(item)


## 4. Install the two CPU runtimes into separate environments

There is no global runtime upgrade/downgrade. The same NumPy version is pinned in both. Native-library hashes, build strings and installed-package lists are recorded. A missing runtime wheel stops the study instead of silently changing its version.


In [ ]:
def save_incomplete(phase,error):
    (OUTPUT/'failure.json').write_text(json.dumps({'phase':phase,'complete':False,'error':str(error)},indent=2))
    analyzer=load_helper('analyze')
    archive=OUTPUT.parent/(OUTPUT.name+'_INCOMPLETE_reports.zip')
    analyzer.pack(OUTPUT,archive)
    print('INCOMPLETE diagnostics saved:',archive)
try:
    ENVIRONMENTS=study.install_environments(CONFIG,OUTPUT)
except Exception as error:
    save_incomplete('runtime_installation',error)
    raise
print('Isolated runtimes are ready. No inference has run yet.')


## 5. Run the complete frozen matrix

This is the long cell. It prints job starts, periodic progress and completion times. Every worker verifies its runtime, graph, inputs and implementation hashes. A fixed subset also checks within-session repeatability. The full suite is required before analysis; failed settings are preserved and are not dropped.

If this cell is interrupted in the **same unchanged host/session**, rerun it to resume cached jobs and partial checkpoints. Preserve output before the Kaggle session expires. A changed host or environment requires a separate study folder.


In [ ]:
try:
    study.execute(CONFIG,OUTPUT,JOBS,ENVIRONMENTS)
except Exception as error:
    save_incomplete('matrix_execution',error)
    raise
print('All declared inference jobs completed.')


## 6. Analyze all settings, including null and adverse results

Reports separate: (a) matched runtime/optimizer/thread contrasts, (b) same-setting FP32/compressed decision changes, (c) source-label outcomes, and (d) agreement with historical Kaggle/Android endpoints.

The min/max envelope describes **only this finite tested suite**. It is not a portability certificate for new phones, runtime versions or inputs. The historical Android endpoint is not a newly controlled phone-version experiment. Host one-pass timings are not the Flutter pipeline measurements.


In [ ]:
analyzer=load_helper('analyze')
try:
    COMPLETION=analyzer.main(OUTPUT,BUNDLE)
except Exception as error:
    save_incomplete('analysis',error)
    raise
assert COMPLETION['all_jobs_complete']
print((OUTPUT/'READ_ME_results.txt').read_text())


## 7. Inspect figures and download the report

`historical_endpoint_comparisons.csv` shows whether the controlled host settings reproduce or narrow the prior disagreement. A host match alone is not proof of a phone fix. `finite_suite_stability.csv` and `common_256_panel.csv` show how widely the behavior repeats across the actual available artifacts. Do not pool seeds, repeated conditions or devices as independent source messages.

The stress partition is reported as **alert rate on keyword-enriched source-negative messages**. No annotators or corrected labels are introduced. The source remains the already inspected Twitter corpus; this notebook does not close dataset-diversity or complete historical-overlap limitations.


In [ ]:
from IPython.display import display,Image,FileLink
for name in ['runtime_decision_matrix.png','finite_suite_primary_instability.png']:
    path=OUTPUT/name
    if path.exists():display(Image(filename=str(path)))
REPORT=OUTPUT.parent/(OUTPUT.name+'_reports.zip')
assert REPORT.exists()
with zipfile.ZipFile(REPORT) as archive:
    assert archive.testzip() is None
    assert not any(name.endswith('.onnx') for name in archive.namelist())
print('RETURN THIS FILE:',REPORT)
print('Report size:',round(REPORT.stat().st_size/1e6,2),'MB')
display(FileLink(str(REPORT)))
print('The device_replay_handoff folder prepares a separate physical-device replay. It is not an executable phone app or a replacement for frozen Flutter assets.')


## What could strengthen the paper

A reproducible one-factor effect across artifacts would support a mechanism/boundary result beyond the original two-device observation. A null host result is also retained: it can show that version number alone does not explain the observed cross-environment difference, while leaving OS/build/hardware causes unresolved. Multiseed claims require actual additional model coverage.

The general possibility of numerical variation is established knowledge. The study's value depends on the new controlled results, not a guarantee of novelty, acceptance, clinical utility or accuracy preservation.

Official controls: [ONNX Runtime threading](https://onnxruntime.ai/docs/performance/tune-performance/threading.html), [session options](https://onnxruntime.ai/docs/api/python/api_summary.html), [quantization debugging](https://onnxruntime.ai/docs/performance/model-optimizations/quantization.html).
